In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/glovetwitter27b100dtxt/glove.twitter.27B.200d.txt
/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/sample_submission.csv
/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/train_dataset.csv
/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/test_dataset.csv
/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/val_dataset.csv


In [2]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("fullmetal26/glovetwitter27b100dtxt")

# print("Path to dataset files:", path)

In [3]:
# # %pip install torch 
# %pip install pandas 
# %pip install gensim 
# %pip install nltk 
# %pip install re
# %pip install matplotlib 
# %pip install random 
# %pip install os 
# %pip install optuna 
# %pip install scikit-learn
# %pip install numpy
# %pip install torch --index-url https://download.pytorch.org/whl/cu121
# # %pip install 
# # !pip install 
# %pip uninstall gensim -y
# %pip install --upgrade pip setuptools wheel
# %pip install gensim

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import gensim
import nltk
import re
import matplotlib.pyplot as plt
import random
import os
import optuna
import sklearn
import numpy as np

# from google.colab import drive
# drive.mount('/content/drive')

from torch.utils.data import Dataset, DataLoader, TensorDataset
from nltk.tokenize import word_tokenize
from gensim.models.keyedvectors import KeyedVectors
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer
from gensim.scripts.glove2word2vec import glove2word2vec



lemmatizer = WordNetLemmatizer()

nltk.download('punkt', download_dir='/kaggle/working/nltk_data')
nltk.download('stopwords', download_dir='/kaggle/working/nltk_data')
nltk.download('wordnet', download_dir='/kaggle/working/nltk_data')
nltk.download('omw-1.4', download_dir='/kaggle/working/nltk_data')

# Tell nltk where to find the data
import os
nltk.data.path.append('/kaggle/working/nltk_data')


[nltk_data] Downloading package punkt to /kaggle/working/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /kaggle/working/nltk_data...


In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# Ensure you have required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

In [8]:
source_dir = r"/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/"
trainData = pd.read_csv(os.path.join(source_dir, "train_dataset.csv"))
testData = pd.read_csv(os.path.join(source_dir, "test_dataset.csv"))
validationData = pd.read_csv(os.path.join(source_dir, "val_dataset.csv"))

trainData = trainData.dropna(subset=['Text', trainData.columns[-1]])
validationData = validationData.dropna(subset=['Text', validationData.columns[-1]])
testData = testData.dropna(subset=['Text', testData.columns[-1]]) # Ας καθαρίσουμε και το test data αν έχει NaN

In [9]:
def text_preprocessing(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)
    text = re.sub(r'pic.\S+', '', text)
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"'re", ' are', text)
    text = re.sub(r"'ll", ' will', text)
    text = re.sub(r"'m", ' am', text)
    text = re.sub(r"'s", ' is', text)
    text = re.sub(r'\babt\b', 'about', text)
    text = re.sub(r'\band\b', 'and', text)
    text = re.sub(r'\bb4\b', 'before', text)
    text = re.sub(r'\bbrb\b', 'be right back', text)
    text = re.sub(r'\bbtw\b', 'but the way', text)
    text = re.sub(r'\bgr8\b', 'great', text)
    text = re.sub(r'\bgud\b', 'good', text)
    text = re.sub(r'\bgn\b', 'good night', text)
    text = re.sub(r'\bh8\b', 'hate', text)
    text = re.sub(r'\bhavent\b', "have not", text)
    text = re.sub(r'\bhavnt\b', "have not", text)
    text = re.sub(r'\bhav\b', 'have', text)
    text = re.sub(r'\bidk\b', "i do not know", text)
    text = re.sub(r'\b2moro\b', 'tomorrow', text)
    text = re.sub(r'\b2nite\b', 'tonight', text)
    text = re.sub(r'\bcant\b', "can not", text)
    text = re.sub(r'\bcnt\b', "can not", text)
    text = re.sub(r'\bcuz\b', 'because', text)
    text = re.sub(r'\bfreind\b', 'friend', text)
    text = re.sub(r'\bfyi\b', 'for your information', text)
    text = re.sub(r'\bgr8\b', 'great', text)
    text = re.sub(r'\bskool\b', 'school', text)
    text = re.sub(r'\bl8r\b', 'later', text)
    text = re.sub(r'\bm8\b', 'mate', text)
    text = re.sub(r'\bmabe\b', 'maybe', text)
    text = re.sub(r'\bn0\b', 'no one', text)
    text = re.sub(r'\bno1\b', 'no one', text)
    text = re.sub(r'\bplz\b', 'please', text)
    text = re.sub(r'\bpls\b', 'please', text)
    text = re.sub(r'\bta\b', 'thanks', text)
    text = re.sub(r'\bthanx\b', 'thanks', text)
    text = re.sub(r'\bteh\b', 'the', text)
    text = re.sub(r'\bthx\b', 'thanks', text)
    text = re.sub(r'\bthier\b', 'their', text)
    text = re.sub(r'\bwut\b', 'what', text)
    text = re.sub(r'\bwould\b', 'would', text)
    text = re.sub(r'\bwierd\b', 'weird', text)
    text = re.sub(r'\bskool\b', 'school', text)
    text = re.sub(r'(?<!\w)u(?!\w)', 'you', text)
    text = re.sub(r'\br\b', 'are', text)
    text = re.sub(r'(?<!\w)y(?!\w)', 'why', text)
    text = re.sub(r'\bthats\b', 'that is', text)
    text = re.sub(r'\bthatis\b', 'that is', text)
    text = re.sub(r'\bcya\b', 'see you', text)
    text = re.sub(r'\bwasnt\b', "was not", text)
    text = re.sub(r'\bkno\b', 'know', text)
    text = re.sub(r'\bur\b', 'you are', text)
    text = re.sub(r'&[^;]+;', '', text)
    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'@#\$%\^&\*\.,\'\""', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)
    return tokens
    # lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # return lemmatized_tokens

trainData['Tokens'] = trainData['Text'].apply(text_preprocessing)
validationData['Tokens'] = validationData['Text'].apply(text_preprocessing)
testData['Tokens'] = testData['Text'].apply(text_preprocessing)

In [10]:
x_df_train = pd.DataFrame(trainData, columns=trainData.columns[:-2])
y_df_train = pd.DataFrame(trainData, columns=[trainData.columns[-2]])

x_df_val = pd.DataFrame(validationData, columns=validationData.columns[:-2])
y_df_val = pd.DataFrame(validationData, columns=[validationData.columns[-2]])

In [11]:
glove_input_file = '/kaggle/input/glovetwitter27b100dtxt/glove.twitter.27B.200d.txt'
w2v_output_file = '/kaggle/working/glv_with_w2v_format.txt'

glove2word2vec(glove_input_file, w2v_output_file) 

word2vec_model = KeyedVectors.load_word2vec_format(w2v_output_file, binary=False)

def tokens_to_vec(tokens, model, vector_size=200):
    vecs = []
    for token in tokens:
        if token in model:
            vecs.append(model[token])
            
    if len(vecs) == 0:
        return np.zeros(vector_size)
    else:
        return np.mean(vecs, axis=0)


x_vectors_train = np.array([tokens_to_vec(tokens, word2vec_model, vector_size=200) for tokens in trainData['Tokens']])
x_vectors_val = np.array([tokens_to_vec(tokens, word2vec_model, vector_size=200) for tokens in validationData['Tokens']])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_vectors_train = scaler.fit_transform(x_vectors_train)  # Train μόνο εδώ
x_vectors_val = scaler.transform(x_vectors_val)    

<ipython-input-11-1d64ed6a6cf3>:4: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, w2v_output_file)


In [12]:
x_train = torch.tensor(x_vectors_train, dtype=torch.float)
y_train = torch.tensor(y_df_train.values.squeeze(), dtype=torch.float)

x_val = torch.tensor(x_vectors_val, dtype=torch.float)
y_val = torch.tensor(y_df_val.values.squeeze(), dtype=torch.float)

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

train_accuracies = []
val_accuracies = []

train_precisions = []
train_recalls = []
train_f1s = []
val_precisions = []
val_recalls = []
val_f1s = []

In [14]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, D_in, H1, H2, D_out, dropout_rate, activation_name="LeakyReLU"):
        super(Net, self).__init__()
        self.input_bn = nn.BatchNorm1d(D_in)

        activation = self._get_activation(activation_name)

        self.block1 = nn.Sequential(
            nn.Linear(D_in, H1),
            nn.BatchNorm1d(H1),
            activation,
            nn.Dropout(dropout_rate)             
        )
        self.block2 = nn.Sequential(
            nn.Linear(H1, H2),
            nn.BatchNorm1d(H2),
            activation,
            nn.Dropout(dropout_rate)
        )

        self.out = nn.Linear(H2, D_out)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x)

    def _get_activation(self, name):
        if name == "LeakyReLU":
            return nn.LeakyReLU(negative_slope=0.01)
        elif name == "ReLU":
            return nn.ReLU()
        elif name == "SELU":
            return nn.SELU()
        else:
            raise ValueError(f"Unsupported activation: {name}")


In [15]:
def objective(trial):
    optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "AdamW"])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.4, 0.5, step=0.05)
    weight_decay = trial.suggest_float("weight_decay", 1e-3, 1e-1, log=True)
    activation_name = trial.suggest_categorical("activation", ["LeakyReLU", "ReLU", "SELU"])
    batch_size = trial.suggest_categorical("batch_size", [64, 128])
    
    D_in = x_train.shape[1]
    # H1 = trial.suggest_int("H1", 64, 256, step=32)
    H1 = trial.suggest_int("H1", 32, 128, step=32)
    H2 = trial.suggest_int("H2", 16, 96, step=16)
    D_out = 1

    model = Net(D_in, H1, H2, D_out, dropout_rate, activation_name)
    loss_func = nn.BCEWithLogitsLoss() 

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    # elif optimizer_name == "SGD":
    #     optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    # elif optimizer_name == "RMSprop":
    #     optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    trainDataset = TensorDataset(x_train, y_train)
    trainDataloader = DataLoader(trainDataset, batch_size=batch_size, shuffle=True)

    valDataset = TensorDataset(x_val, y_val)
    valDataloader = DataLoader(valDataset, batch_size=batch_size, shuffle=False) 

    best_val_loss = float('inf')
    patience = 10
    trigger_times = 0
    max_epochs = 50

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5) # 'min' για loss

    for epoch in range(max_epochs):  
        model.train()
        total_train_loss = 0
        for x_batch, y_batch in trainDataloader:
            y_pred = model(x_batch).squeeze()
            y_batch = y_batch.squeeze()
            loss = loss_func(y_pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(trainDataloader)

        # Validation phase
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for x_val_batch, y_val_batch in valDataloader:
                y_val_pred = model(x_val_batch).squeeze()
                val_loss = loss_func(y_val_pred, y_val_batch)
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(valDataloader)

        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                break 


    model.eval()
    with torch.no_grad():
        y_pred_logits_train = model(x_train).squeeze()
        y_pred_labels_train = (torch.sigmoid(y_pred_logits_train) > 0.5).float()

    train_acc = accuracy_score(y_train.cpu().numpy(), y_pred_labels_train.cpu().numpy())

    model.eval()
    with torch.no_grad():
        y_pred_logits_val = model(x_val).squeeze()
        y_pred_labels_val = (torch.sigmoid(y_pred_logits_val) > 0.5).float()

    val_acc = accuracy_score(y_val.cpu().numpy(), y_pred_labels_val.cpu().numpy())
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    val_precision = precision_score(y_val.cpu().numpy(), y_pred_labels_val.cpu().numpy(), zero_division=0)
    val_recall = recall_score(y_val.cpu().numpy(), y_pred_labels_val.cpu().numpy(), zero_division=0)
    val_f1 = f1_score(y_val.cpu().numpy(), y_pred_labels_val.cpu().numpy(), zero_division=0)

    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)

    print(f"Trial {trial.number} - "
        f"Train Accuracy: {train_acc:.8f}, "
        f"Val Accuracy: {val_acc:.8f}, "
        f"Val Precision: {val_precision:.8f}, "
        f"Val Recall: {val_recall:.8f}, "
        f"Val F1: {val_f1:.8f}")


    return 1.0 - val_acc 



In [16]:
# Optuna study
# optuna_sample = optuna.create_study(direction='minimize', study_name='accuracy_optimization')
# optuna_sample.optimize(objective, n_trials = 8)
    
# print('Number of finished trials:', len(optuna_sample.trials))
# print('Best hyperparameters:', optuna_sample.best_trial.params)  
# print('Best accuracy:', 1.0 - optuna_sample.best_value)

# best_params = optuna_sample.best_trial.params

In [17]:
best_params = {'optimizer_name': 'AdamW', 'learning_rate': 0.004169673329375278, 'dropout_rate': 0.45, 'weight_decay': 0.0010285156143153202, 'activation': 'LeakyReLU', 'batch_size': 128, 'H1': 32, 'H2': 96}

print("best_params: ", best_params)

best_params:  {'optimizer_name': 'AdamW', 'learning_rate': 0.004169673329375278, 'dropout_rate': 0.45, 'weight_decay': 0.0010285156143153202, 'activation': 'LeakyReLU', 'batch_size': 128, 'H1': 32, 'H2': 96}


In [18]:
x_vectors_test = np.array([tokens_to_vec(tokens, word2vec_model, vector_size=200) for tokens in testData['Tokens']])
x_vectors_test = scaler.transform(x_vectors_test)
x_test = torch.tensor(x_vectors_test, dtype=torch.float)

model = Net(
    D_in=x_train.shape[1],
    H1=best_params['H1'],
    H2=best_params['H2'],
    D_out=1,
    dropout_rate=best_params['dropout_rate'],
    activation_name=best_params['activation']
)

loss_func = nn.BCEWithLogitsLoss()

if best_params['optimizer_name'] == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_params['learning_rate'])

trainDataset = TensorDataset(x_train, y_train)
trainDataloader = DataLoader(trainDataset, batch_size=best_params['batch_size'], shuffle=True)

epochs = 50
model.train()
for epoch in range(epochs):
    total_loss = 0
    for xb, yb in trainDataloader:
        pred = model(xb).squeeze()
        loss = loss_func(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

model.eval()
with torch.no_grad():
    y_pred_logits_test = model(x_test).squeeze()
    y_pred_probs_test = torch.sigmoid(y_pred_logits_test)
    y_pred_labels_test = (y_pred_probs_test > 0.5).float()

id_column = None
for column in testData.columns:
    if column.lower() == 'id':
        id_column = column
        break

if id_column:
    submission = pd.DataFrame({
        'ID': testData[id_column], 
        'Label': y_pred_labels_test.cpu().numpy().astype(int)
    })
else:
    print("No 'id' column found. Using index as ID.")
    submission = pd.DataFrame({
        'ID': range(len(y_pred_labels_test)),
        'Label': y_pred_labels_test.cpu().numpy().astype(int)
    })

submission.to_csv('submission.csv', index=False)